# Qwen2.5-7B 环境检查

这个 notebook 用来确认当前环境是否适合继续运行 Qwen2.5-7B 的训练、量化和部署流程。

环境检查的目的不是装饰步骤。大模型项目里很多错误来自版本不匹配、CUDA 不可用、GPU 显存不足、bitsandbytes 或 flash-attn 没装好。先检查环境，可以避免后面在训练单元格里才发现问题。

## 1. 检查 Python 和系统信息

先确认 Python 版本、操作系统和当前工作目录。Python 版本过低可能导致 Transformers、TRL、PEFT 等库无法安装或行为异常。

In [ ]:
import os
import platform
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", os.getcwd())

## 2. 检查 PyTorch 和 GPU

Qwen2.5-7B 做 LoRA/QLoRA 训练通常需要 GPU。没有 GPU 也可以阅读 notebook，但不建议直接运行训练部分。

重点看三个结果：

- `torch.cuda.is_available()`：是否能使用 CUDA。
- GPU 名称：判断显卡级别。
- 显存大小：决定能否加载模型、能用多长上下文、batch size 应该设多小。

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        total_gb = props.total_memory / 1024**3
        print(f"GPU {index}: {props.name}, {total_gb:.1f} GB")

## 3. 检查核心 Python 包

这些包分别负责不同环节：

- `transformers`：模型和 tokenizer 加载。
- `datasets`：读取 JSONL 数据。
- `trl`：SFT、DPO、PPO、GRPO 等训练器。
- `peft`：LoRA、QLoRA adapter。
- `accelerate`：多设备和混合精度训练支持。
- `bitsandbytes`：4bit/8bit 加载，QLoRA 常用。

Qwen2.5 需要较新的 Transformers。版本过低时可能无法识别 Qwen2 架构。

In [ ]:
from importlib.metadata import PackageNotFoundError, version

packages = [
    "transformers",
    "datasets",
    "trl",
    "peft",
    "accelerate",
    "bitsandbytes",
]

for package in packages:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: not installed")

## 4. 检查训练和部署相关命令

后续部署会用到 vLLM、SGLang、Ollama、llama.cpp 等工具。这里不强制要求都安装，只确认当前机器能找到哪些命令。

In [ ]:
import shutil

commands = ["vllm", "sglang", "ollama", "llama-cli", "llama-server"]
for command in commands:
    path = shutil.which(command)
    print(f"{command}: {path if path else 'not found'}")

## 5. 快速判断

如果你只是阅读内容：不需要 GPU，也不需要装齐所有包。

如果你要运行 Qwen2.5-7B LoRA：建议至少有一张 16GB 以上显存的 GPU，并使用 bf16 或 fp16。

如果你要运行 QLoRA：需要 `bitsandbytes` 正常工作，显存要求会更低，但训练速度可能更慢。

如果你要部署服务：Transformers 最适合先验证，vLLM 适合后续服务化，Ollama/llama.cpp 适合 GGUF 本地运行。